# Portfolio ML Pipeline on Colab

This notebook uses:
- **GitHub** for the latest code
- **Google Drive** for local CSV data, logs, and outputs

Persistence model:
- Code is re-cloned from GitHub each session.
- Input CSVs are copied from Drive into the repo before each run.
- Logs and outputs are written back to Drive.
- If Colab disconnects during a run, you do not get true mid-command resume, but all completed artifacts remain in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/portfolio_ml_project')
DRIVE_DATA = DRIVE_ROOT / 'data_local'
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
DRIVE_LOGS = DRIVE_ROOT / 'logs'

for path in [DRIVE_ROOT, DRIVE_DATA, DRIVE_OUTPUTS, DRIVE_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Put these files under Drive data_local:')
print(' - sp500_stocks.csv')
print(' - sp500_companies.csv')
print(' - SP500.csv')

## Set your GitHub repo URL

Replace the URL below with your pushed repository.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/BhavyaLikhitha/Portfolio-Optimization-and-Market-Performance-Analysis.git'
REPO_DIR = '/content/Portfolio-Optimization-and-Market-Performance-Analysis'

In [ ]:
!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!ls

In [ ]:
import shutil
from pathlib import Path

repo_local = Path(REPO_DIR) / 'data' / 'local'
repo_local.mkdir(parents=True, exist_ok=True)

for filename in ['sp500_stocks.csv', 'sp500_companies.csv', 'SP500.csv']:
    src = DRIVE_DATA / filename
    dst = repo_local / filename
    if not src.exists():
        raise FileNotFoundError(f'Missing required Drive data file: {src}')
    shutil.copy2(src, dst)
    print(f'Copied {src} -> {dst}')

In [ ]:
!python --version
!pip install --upgrade pip
!pip install pandas numpy matplotlib seaborn plotly scikit-learn scipy xgboost lightgbm torch shap mlflow streamlit yfinance kagglehub lxml html5lib beautifulsoup4

In [ ]:
!python -m py_compile main.py app.py
!python -c "import main; print('main import ok')"

## Smoke run

In [ ]:
!python main.py --max-tickers 10 --dl-epochs 5 2>&1 | tee /content/drive/MyDrive/portfolio_ml_project/logs/smoke_run.log

In [ ]:
import shutil
from pathlib import Path

src_outputs = Path(REPO_DIR) / 'outputs'
dst_outputs = DRIVE_OUTPUTS
if dst_outputs.exists():
    shutil.rmtree(dst_outputs)
if src_outputs.exists():
    shutil.copytree(src_outputs, dst_outputs)
print('Outputs synced to Drive:', dst_outputs)

## Fuller run

In [ ]:
!python main.py --max-tickers 25 --dl-epochs 10 2>&1 | tee /content/drive/MyDrive/portfolio_ml_project/logs/full_run.log

In [ ]:
import shutil
from pathlib import Path

src_outputs = Path(REPO_DIR) / 'outputs'
dst_outputs = DRIVE_OUTPUTS
if dst_outputs.exists():
    shutil.rmtree(dst_outputs)
if src_outputs.exists():
    shutil.copytree(src_outputs, dst_outputs)
print('Outputs synced to Drive:', dst_outputs)

In [ ]:
import json
from pathlib import Path

summary_path = Path(REPO_DIR) / 'outputs' / 'pipeline_summary.json'
if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    summary
else:
    print('No pipeline_summary.json found yet.')